# Real-time Banking Fraud Detection — Live Dashboard

This notebook is the **monitoring dashboard** for Bank X's fraud-detection platform.

It reads the Parquet files continuously produced by the **Spark Streaming processor**
(shared `/data/output` volume) and refreshes automatically.

It displays, for the **last 20 users** seen in transactions:

- Average amounts sent / received over the **3h / 7d / 3w / 3m** sliding windows
- Transaction counts and distinct network size (senders / receivers) per window
- Lifetime ("since account creation") totals and periodic averages
- Activity over the **last 10 seconds**
- Charts and **red highlighting** for anomalous spending spikes

**How to use:** run all cells top to bottom. The last cell starts the auto-refresh
loop — interrupt the kernel (■ stop button) to stop it.


In [1]:
# --- Configuration and imports --------------------------------------------
import os
import time
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

warnings.filterwarnings("ignore")

# The Spark processor writes its results here (shared Docker volume).
OUTPUT_DIR   = os.environ.get("DATA_DIR", "/workspace/data") + "/output"
USER_METRICS = OUTPUT_DIR + "/user_metrics"
RECENT_TX    = OUTPUT_DIR + "/recent_tx"

REFRESH_SECONDS = 5     # auto-refresh interval
ITERATIONS      = 120   # number of refreshes (120 x 5s = 10 minutes)
N_USERS         = 20    # show the last N users that appeared
ACTIVITY_WINDOW = 10    # "last 10 seconds of activity"

print("Dashboard configured. Reading Spark output from:", OUTPUT_DIR)


Dashboard configured. Reading Spark output from: /workspace/data/output


In [2]:
# --- Helper functions ------------------------------------------------------
def load_parquet(path):
    # Read a Parquet directory written by Spark.
    # The processor overwrites these directories every few seconds, so a read
    # may occasionally land mid-write -> we simply retry a couple of times.
    for _ in range(3):
        try:
            return pd.read_parquet(path)
        except Exception:
            time.sleep(0.5)
    return None


def get_recent_users(recent, n):
    # Return the last `n` distinct users that appeared in transactions,
    # counting both senders and receivers, most-recent first.
    events = pd.concat([
        recent[["ts", "send_id"]].rename(columns={"send_id": "user"}),
        recent[["ts", "receive_id"]].rename(columns={"receive_id": "user"}),
    ])
    events = events.sort_values("ts", ascending=False)
    return list(dict.fromkeys(events["user"]))[:n]


def highlight_anomalies(row):
    # pandas Styler callback: paint a row red when it is flagged anomalous.
    spike = bool(row.get("anomaly", False))
    return ["background-color: #ffcccc" if spike else "" for _ in row]


In [ ]:
# --- The dashboard renderer ------------------------------------------------
def render():
    recent  = load_parquet(RECENT_TX)
    print("loaded the parquet")
    metrics = load_parquet(USER_METRICS)

    clear_output(wait=True)
    print("=" * 78)
    print("   REAL-TIME BANKING FRAUD DETECTION  -  Bank X monitoring committee")
    print("   refreshed at", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print("=" * 78)

    if metrics is None or len(metrics) == 0:
        print("\nWaiting for the Spark processor's first batch...")
        print("Open a Jupyter terminal and run:  python processor/processor.py")
        return
    # If recent_tx is briefly empty (processor still draining backlog), we keep
    # showing the per-user metrics table instead of blanking the dashboard.
    no_recent = recent is None or len(recent) == 0

    if not no_recent:
        recent["ts"] = pd.to_datetime(recent["ts"])
        now = recent["ts"].max()
        last_window = recent[recent["ts"] >= now - timedelta(seconds=ACTIVITY_WINDOW)]
        tps = len(last_window) / ACTIVITY_WINDOW
        print(f"\nActivity in the last {ACTIVITY_WINDOW}s : {len(last_window)} "
              f"transactions   (~{tps:.1f} tx/s)")
        print(f"Distinct users currently tracked by Spark : {len(metrics)}")
        users = get_recent_users(recent, N_USERS)
        view = metrics[metrics["user"].isin(users)].copy()
        view["__order"] = view["user"].apply(lambda u: users.index(u))
        view = view.sort_values("__order").drop(columns="__order")
    else:
        print(f"\n[ recent_tx is briefly empty - processor catching up to live data ]")
        print(f"Distinct users currently tracked by Spark : {len(metrics)}")
        # Show the most active users by 3h send count instead.
        view = metrics.dropna(subset=["count_sent_3h"]).sort_values(
            "count_sent_3h", ascending=False).head(N_USERS).copy()
        last_window = None

    # ---- Fraud detection: three USER-RELATIVE rules ------------------------
    # Every threshold below is expressed against this user's OWN history,
    # not a global magic number. A user who normally sends 5 MRU per tx and
    # suddenly sends 50 is flagged; a user who normally sends 50 and sends
    # 60 is not.
    avg_3h   = view["avg_amount_sent_3h"].fillna(0)
    avg_life = view["avg_amount_sent_life"].fillna(0)
    cnt_3h   = view["count_sent_3h"].fillna(0)
    cnt_7d   = view["count_sent_7d"].fillna(0)
    dist_3h  = view["distinct_sent_3h"].fillna(0)
    dist_life = view["distinct_sent_life"].fillna(0)

    # Expected 3h transaction count derived from the user's own 7d rate:
    # if you usually send 56 tx per week, you usually send ~1 in any 3h slice.
    expected_3h_cnt = cnt_7d * (3.0 / 168.0)

    # Rule A - AMOUNT SPIKE: 3h average amount is more than 3x this user's
    #   own lifetime average (with at least 3 recent transactions to be sure).
    ratio_amount = (avg_3h / avg_life.replace(0, np.nan)).fillna(0)
    flag_A = (ratio_amount > 3) & (cnt_3h >= 3)

    # Rule B - NETWORK FAN-OUT: the user reached >= 50% of their lifetime
    #   distinct receivers in just the last 3h, with non-trivial amounts
    #   relative to their own average.
    ratio_network = (dist_3h / dist_life.replace(0, np.nan)).fillna(0)
    flag_B = (
        (ratio_network >= 0.5) &
        (dist_3h >= 5) &
        (avg_3h >= avg_life) &
        (dist_life > 0)
    )

    # Rule C - ACTIVITY BURST: the 3h count is > 5x the count we would
    #   expect given the user's own 7d-rate baseline.
    ratio_burst = (cnt_3h / expected_3h_cnt.replace(0, np.nan)).fillna(0)
    flag_C = (
        (ratio_burst > 5) &
        (cnt_3h >= 5) &
        (avg_3h >= avg_life) &
        (cnt_7d > 0)
    )

    view["flag_amount"]  = flag_A
    view["flag_network"] = flag_B
    view["flag_burst"]   = flag_C
    view["anomaly"] = flag_A | flag_B | flag_C
    view["fraud_reasons"] = (
        flag_A.map({True: "AMOUNT ", False: ""}) +
        flag_B.map({True: "NETWORK ", False: ""}) +
        flag_C.map({True: "BURST", False: ""})
    ).str.strip()
    # Ratios shown in the alert table so a human can see WHY the row fired.
    view["x_amount_vs_life"]  = ratio_amount.round(2)
    view["x_network_vs_life"] = ratio_network.round(2)
    view["x_burst_vs_7d"]     = ratio_burst.round(2)

    # ---- Top of dashboard: FRAUD ALERTS panel ------------------------------
    n_anom = int(view["anomaly"].sum())
    if n_anom:
        print(f"\n*** FRAUD ALERTS  ({n_anom} user(s) flagged) ***")
        alerts = (view[view["anomaly"]]
                  [["user", "bank", "fraud_reasons",
                    "count_sent_3h", "avg_amount_sent_3h", "avg_amount_sent_life",
                    "x_amount_vs_life", "distinct_sent_3h", "distinct_sent_life",
                    "x_network_vs_life", "x_burst_vs_7d"]]
                  .fillna(0).round(2)
                  .rename(columns={
                      "avg_amount_sent_3h":   "avg_amt_3h",
                      "avg_amount_sent_life": "avg_amt_life",
                      "count_sent_3h":        "tx_cnt_3h",
                      "distinct_sent_3h":     "dist_recv_3h",
                      "distinct_sent_life":   "dist_recv_life",
                      "fraud_reasons":        "triggered_rules",
                      "x_amount_vs_life":     "x_amt/life",
                      "x_network_vs_life":    "x_dist/life",
                      "x_burst_vs_7d":        "x_cnt/expected",
                  }))
        display(alerts.style.set_properties(**{"background-color": "#ffd6d6"})
                .hide(axis="index"))
    else:
        print("\n[ No fraud alerts among the displayed users ]")

    # ---- Table 1 : average amounts per window -----------------------------
    print(f"\n--- AVERAGE AMOUNTS  (sent / received)  -  last {N_USERS} users ---")
    cols = ["user", "bank"]
    for w in ["3h", "7d", "3w", "3m", "life"]:
        cols += [f"avg_amount_sent_{w}", f"avg_amount_recv_{w}"]
    t1 = view[cols + ["anomaly"]].fillna(0).round(2)
    display(t1.style.apply(highlight_anomalies, axis=1).hide(axis="index"))

    # ---- Table 2 : counts and distinct network per window -----------------
    print(f"\n--- TRANSACTION COUNTS & DISTINCT NETWORK  -  last {N_USERS} users ---")
    cols2 = ["user", "bank"]
    for w in ["3h", "7d", "3w", "3m", "life"]:
        cols2 += [f"count_sent_{w}", f"count_recv_{w}",
                  f"distinct_sent_{w}", f"distinct_recv_{w}"]
    t2 = view[cols2 + ["anomaly"]].fillna(0)
    display(t2.style.apply(highlight_anomalies, axis=1).hide(axis="index"))

    # ---- Table 3 : since account creation ---------------------------------
    print(f"\n--- SINCE ACCOUNT CREATION (lifetime)  -  last {N_USERS} users ---")
    cols3 = ["user", "bank", "total_sent", "total_recv",
             "count_sent_life", "count_recv_life",
             "distinct_sent_life", "distinct_recv_life",
             "avg_hourly_sent", "avg_daily_sent",
             "avg_weekly_sent", "avg_monthly_sent"]
    t3 = view[cols3].fillna(0).round(2)
    display(t3.style.hide(axis="index"))

    # ---- Charts -----------------------------------------------------------
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))

    if not no_recent:
        per_sec = (recent[recent["ts"] >= now - timedelta(seconds=60)]
                   .set_index("ts").resample("1s").size())
        ax[0].plot(per_sec.index, per_sec.values, marker="o", color="steelblue")
        ax[0].set_title("Transactions per second (last 60s)")
    else:
        ax[0].text(0.5, 0.5, "recent_tx not yet available\n(processor catching up)",
                   ha="center", va="center", transform=ax[0].transAxes,
                   color="gray")
        ax[0].set_title("Transactions per second (last 60s)")
    ax[0].set_xlabel("time")
    ax[0].set_ylabel("tx/s")
    ax[0].tick_params(axis="x", rotation=45)
    ax[0].grid(alpha=0.3)

    top = view.sort_values("count_sent_3h", ascending=False).head(10)
    colors = ["red" if a else "steelblue" for a in top["anomaly"]]
    ax[1].barh(top["user"].astype(str), top["count_sent_3h"].fillna(0), color=colors)
    ax[1].set_title("Top displayed users by tx sent (3h window)")
    ax[1].set_xlabel("transactions sent")
    ax[1].invert_yaxis()

    plt.tight_layout()
    plt.show()

    n_anom = int(view["anomaly"].sum())
    if n_anom:
        print(f"\n[!] {n_anom} user(s) flagged by fraud rules (red rows). "
              f"Rules triggered: AMOUNT spike | NETWORK fan-out | activity BURST.")
    else:
        print("\nNo anomalies among the displayed users.")


## Live dashboard

Run the cell below to start the auto-refreshing dashboard.

- It refreshes every **5 seconds** and reads the latest Spark output each time.
- Rows highlighted in **red** are flagged by one of three **user-relative**
  fraud rules — every threshold is expressed against the user's *own* history,
  not a global magic number:
  - **AMOUNT** — 3h average amount is more than **3x** the user's own lifetime average (≥ 3 tx in 3h)
  - **NETWORK** — the user reached **≥ 50%** of their lifetime distinct receivers in just 3h, with amounts at least at their lifetime average
  - **BURST** — 3h tx count is **> 5x** what's expected from the user's own 7-day rate, with amounts at least at their lifetime average
- The FRAUD ALERTS panel shows the ratios that fired (`x_amt/life`, `x_dist/life`, `x_cnt/expected`) so you can see *why* each row was flagged.
- Stop it any time with the kernel **interrupt / stop** button.


In [5]:
# --- Start the live auto-refreshing dashboard ------------------------------
# Interrupt the kernel (the square stop button) to stop the loop.
try:
    for i in range(ITERATIONS):
        render()
        print(f"\n(refresh {i + 1}/{ITERATIONS}  -  next update in "
              f"{REFRESH_SECONDS}s  -  interrupt the kernel to stop)")
        time.sleep(5)
    print("Reached the configured number of refreshes. Re-run this cell to continue.")
except KeyboardInterrupt:
    print("Dashboard stopped by user.")


   REAL-TIME BANKING FRAUD DETECTION  -  Bank X monitoring committee
   refreshed at 2026-05-21 16:21:44

[ recent_tx is briefly empty - processor catching up to live data ]
Distinct users currently tracked by Spark : 247268

*** FRAUD ALERTS  (20 user(s) flagged) ***


user,bank,triggered_rules,tx_count_3h,avg_amount_3h,distinct_recv_3h,avg_amount_lifetime
user_0294019,bank_B,NETWORK BURST,42.000000,44.880000,22.000000,44.880000
user_0036610,bank_X,NETWORK BURST,42.000000,152.010000,22.000000,152.010000
user_0080349,bank_X,NETWORK BURST,41.000000,212.880000,21.000000,208.540000
user_0101685,bank_A,NETWORK BURST,41.000000,66.980000,21.000000,66.980000
user_0179533,bank_A,NETWORK BURST,40.000000,155.170000,20.000000,148.640000
user_0182495,bank_A,NETWORK BURST,40.000000,516.730000,20.000000,516.730000
user_0114428,bank_A,NETWORK BURST,40.000000,44.950000,20.000000,44.950000
user_0226619,bank_B,NETWORK BURST,40.000000,103.560000,20.000000,103.560000
user_0275149,bank_B,NETWORK BURST,40.000000,606.050000,20.000000,606.050000
user_0207967,bank_B,NETWORK BURST,25.000000,70.800000,23.000000,70.800000



--- AVERAGE AMOUNTS  (sent / received)  -  last 20 users ---


user,bank,avg_amount_sent_3h,avg_amount_recv_3h,avg_amount_sent_7d,avg_amount_recv_7d,avg_amount_sent_3w,avg_amount_recv_3w,avg_amount_sent_3m,avg_amount_recv_3m,avg_amount_sent_life,avg_amount_recv_life,anomaly
user_0294019,bank_B,44.880000,10.440000,44.880000,10.440000,44.880000,10.440000,44.880000,10.440000,44.880000,10.440000,True
user_0036610,bank_X,152.010000,3.400000,152.010000,3.400000,152.010000,3.400000,152.010000,3.400000,152.010000,3.400000,True
user_0080349,bank_X,212.880000,2.110000,212.880000,2.110000,212.880000,2.110000,208.540000,2.110000,208.540000,2.110000,True
user_0101685,bank_A,66.980000,0.000000,66.980000,0.000000,66.980000,51.100000,66.980000,51.100000,66.980000,51.100000,True
user_0179533,bank_A,155.170000,14.860000,155.170000,14.860000,155.170000,14.500000,148.640000,14.500000,148.640000,14.500000,True
user_0182495,bank_A,516.730000,0.000000,516.730000,0.000000,516.730000,0.000000,516.730000,0.000000,516.730000,0.000000,True
user_0114428,bank_A,44.950000,0.000000,44.950000,0.000000,44.950000,0.000000,44.950000,10.330000,44.950000,10.330000,True
user_0226619,bank_B,103.560000,23.590000,103.560000,23.590000,103.560000,23.590000,103.560000,23.590000,103.560000,23.590000,True
user_0275149,bank_B,606.050000,7.680000,606.050000,7.680000,606.050000,7.680000,606.050000,7.680000,606.050000,7.680000,True
user_0207967,bank_B,70.800000,0.000000,70.800000,0.000000,70.800000,0.000000,70.800000,0.000000,70.800000,0.000000,True



--- TRANSACTION COUNTS & DISTINCT NETWORK  -  last 20 users ---


user,bank,count_sent_3h,count_recv_3h,distinct_sent_3h,distinct_recv_3h,count_sent_7d,count_recv_7d,distinct_sent_7d,distinct_recv_7d,count_sent_3w,count_recv_3w,distinct_sent_3w,distinct_recv_3w,count_sent_3m,count_recv_3m,distinct_sent_3m,distinct_recv_3m,count_sent_life,count_recv_life,distinct_sent_life,distinct_recv_life,anomaly
user_0294019,bank_B,42.000000,1.000000,22.000000,1.000000,42.000000,1.000000,22.000000,1.000000,42.000000,1.000000,22.000000,1.000000,42.000000,1.000000,22.000000,1.000000,42.000000,1.000000,22.000000,1.000000,True
user_0036610,bank_X,42.000000,4.000000,22.000000,3.000000,42.000000,4.000000,22.000000,3.000000,42.000000,4.000000,22.000000,3.000000,42.000000,4.000000,22.000000,3.000000,42.000000,4.000000,22.000000,3.000000,True
user_0080349,bank_X,41.000000,1.000000,21.000000,1.000000,41.000000,1.000000,21.000000,1.000000,41.000000,1.000000,21.000000,1.000000,42.000000,1.000000,22.000000,1.000000,42.000000,1.000000,22.000000,1.000000,True
user_0101685,bank_A,41.000000,0.000000,21.000000,0.000000,41.000000,0.000000,21.000000,0.000000,41.000000,1.000000,21.000000,1.000000,41.000000,1.000000,21.000000,1.000000,41.000000,1.000000,21.000000,1.000000,True
user_0179533,bank_A,40.000000,2.000000,20.000000,2.000000,40.000000,2.000000,20.000000,2.000000,40.000000,3.000000,20.000000,3.000000,42.000000,3.000000,22.000000,3.000000,42.000000,3.000000,22.000000,3.000000,True
user_0182495,bank_A,40.000000,0.000000,20.000000,0.000000,40.000000,0.000000,20.000000,0.000000,40.000000,0.000000,20.000000,0.000000,40.000000,0.000000,20.000000,0.000000,40.000000,0.000000,20.000000,0.000000,True
user_0114428,bank_A,40.000000,0.000000,20.000000,0.000000,40.000000,0.000000,20.000000,0.000000,40.000000,0.000000,20.000000,0.000000,40.000000,1.000000,20.000000,1.000000,40.000000,1.000000,20.000000,1.000000,True
user_0226619,bank_B,40.000000,1.000000,20.000000,1.000000,40.000000,1.000000,20.000000,1.000000,40.000000,1.000000,20.000000,1.000000,40.000000,1.000000,20.000000,1.000000,40.000000,1.000000,20.000000,1.000000,True
user_0275149,bank_B,40.000000,1.000000,20.000000,1.000000,40.000000,1.000000,20.000000,1.000000,40.000000,1.000000,20.000000,1.000000,40.000000,1.000000,20.000000,1.000000,40.000000,1.000000,20.000000,1.000000,True
user_0207967,bank_B,25.000000,0.000000,23.000000,0.000000,25.000000,0.000000,23.000000,0.000000,25.000000,0.000000,23.000000,0.000000,25.000000,0.000000,23.000000,0.000000,25.000000,0.000000,23.000000,0.000000,True



--- SINCE ACCOUNT CREATION (lifetime)  -  last 20 users ---


user,bank,total_sent,total_recv,count_sent_life,count_recv_life,distinct_sent_life,distinct_recv_life,avg_hourly_sent,avg_daily_sent,avg_weekly_sent,avg_monthly_sent
user_0294019,bank_B,1885.090000,10.440000,42.000000,1.000000,22.000000,1.000000,1885.090000,45242.160000,316695.120000,1357264.800000
user_0036610,bank_X,6384.440000,13.600000,42.000000,4.000000,22.000000,3.000000,6384.440000,153226.560000,1072585.920000,4596796.800000
user_0080349,bank_X,8758.570000,2.110000,42.000000,1.000000,22.000000,1.000000,10.480000,251.500000,1760.490000,7544.970000
user_0101685,bank_A,2746.060000,51.100000,41.000000,1.000000,21.000000,1.000000,7.820000,187.660000,1313.600000,5629.700000
user_0179533,bank_A,6242.870000,43.510000,42.000000,3.000000,22.000000,3.000000,4.290000,103.020000,721.160000,3090.670000
user_0182495,bank_A,20669.160000,0.000000,40.000000,0.000000,20.000000,0.000000,20669.160000,496059.840000,3472418.880000,14881795.200000
user_0114428,bank_A,1798.180000,10.330000,40.000000,1.000000,20.000000,1.000000,0.960000,23.100000,161.730000,693.120000
user_0226619,bank_B,4142.320000,23.590000,40.000000,1.000000,20.000000,1.000000,4142.320000,99415.680000,695909.760000,2982470.400000
user_0275149,bank_B,24241.880000,7.680000,40.000000,1.000000,20.000000,1.000000,24241.880000,581805.120000,4072635.840000,17454153.600000
user_0207967,bank_B,1770.000000,0.000000,25.000000,0.000000,23.000000,0.000000,1770.000000,42480.000000,297360.000000,1274400.000000


Dashboard stopped by user.
